# The Full GPT Block

Running example throughout:
| Symbol | Value | Kind  | What it counts                                |
| ------ | ----- | ----- | --------------------------------------------- |
| B      | 2     | Data  | How many sequences share one forward pass     |
| T      | 4     | Data  | How many positions each sequence has          |
| V      | 65    | Model | How many distinct characters exist            |
| d      | 8     | Model | Width of the residual stream                  |
| h      | 2     | Model | How many separate averages attention computes |
| d_head | 4     | Model | Width of each average                         |
| d_ff   | 32    | Model | Width of the per-position MLP                 |
| L      | 2     | Model | How many times the block is stacked           |

with `h * d_head = 2 * 4 = 8 = d`.

Four characters sit in a row. Each holds a slip of 8 numbers.

To use context, each character does three things with that slip:

- Write a question (what am I looking for?)
- Pin a label on itself (what do I contain?)
- Hold a payload (what should someone receive if they pick me?)

Matching a question against every label produces a score. Softmax turns scores into weights that add up to 1. The new information is the weighted average of the payloads.

One set of question/label/payload maps can only make one average, because the weights on a row must sum to 1. A second head is a second, separate average, with its own maps. The two 4-number results are stacked back into 8 numbers and added onto the original slip.

Token ids have shape `[B, T] = [2, 4]`, each entry an integer in `0 ... 64`.

The embedding table is `[V, d] = [65, 8]`. Looking up four characters in that table gives:

```
X: [B, T, d] = [2, 4, 8]
```

That tensor is the residual stream. Every later block reads it and adds something of the same shape. Nothing in the middle is allowed to change B, T, or d, or the add would not fit.

At the end, a matrix `[d, V] = [8, 65]` scores every character. Logits are `[2, 4, 65]`. Softmax over the last axis is “which character comes next?”

V only appears in those two matrices. Attention never sees 65.

**One head, with these exact sizes**

Take one sequence and one head. X for that sequence is `[4, 8]`.

Three learned maps, each `[d, d_head] = [8, 4]`, build:

```
Q, K, V:  [T, d_head] = [4, 4]
```

## Init

In [1]:
from pathlib import Path
import numpy as np
import torch
from torch import nn

## The Map

Each token is a person at a desk with a notepad of 8 numbers (the *residual stream*, `d=8`). Each round has two phases:
1. **Meeting (attention)**: Everyone may read colleagues' notes, but only from earlier seats.
2. **Private thinking (MLP)**: Everyone works alone at their desk with the same toolkit.

The rules are that you only add to your notepad and never erase it, and before reading a note you rescale a copy to a standard size (LayerNorm).
Seat numbers get stamped on at the start (positions). Each section below maps its piece onto this picture and says where the picture fails.

## Multi-head Attention

**Components**. The reviewers are the heads, each with its own `Wq, Wk, Wv` slice. Stapling is concatenation, and the editor is `Wo`.

**Technical detail**. `X: (2,4,8)`. Compute `Q = X @ Wq` with `Wq: (8, 8)`, likewise `K, V`, each `(2,4,8)`. Then:

|Step|Shape|
|-|-|
|reshape to `(B,T,h,d_head)`|`(2,4,2,4)`|
|transpose axes 1 and 2|`(2,2,4,4)` = `(B,h,T,d_head)`|
|scores `Q@Kᵀ/sqrt(d_head)`|`(2,2,4,4)` = `(B,h,T,T)`|
|causal mask (-inf above the diagonal), softmax|same; rows sum to 1|
|`P @ V`|`(B,h,T,d_head)`|
|transpose back, reshape|`(2,4,8)`|
|`@ Wo`|`(2,4,8)`|

In [2]:
def attention_forward(X, Wq, Wk, Wv, Wo, h, d_head, scale=None):
    Q = X @ Wq  # (B,T,d) = (2,4,8)
    K = X @ Wk  # (B,T,d) = (2,4,8)
    V = X @ Wv  # (B,T,d) = (2,4,8)

    B, T = X.shape[0], X.shape[1]
    d = h * d_head
    d_k = Wk.shape[1]

    # Reshape to (B,T,h,d_head) > transpose axes 1 and 2
    # (B,h,T,d_head) = (2,2,4,4)
    Q = Q.reshape(B, T, h, d_head).transpose(0, 2, 1, 3)
    K = K.reshape(B, T, h, d_head).transpose(0, 2, 1, 3)
    V = V.reshape(B, T, h, d_head).transpose(0, 2, 1, 3)

    # Score - (B,h,T,d_head) @ (B,h,T,d_head)^T = (B,h,Tq,Tk) = (2,2,4,4)
    S = Q @ K.transpose(0, 1, 3, 2)
    if scale is None:
        scale = 1 / np.sqrt(d_k)
    S = S * scale

    # -inf mask
    mask = np.triu(np.ones((T, T), dtype=bool), k=1)
    S = np.where(mask, -np.inf, S)

    # Stability
    S -= S.max(axis=-1, keepdims=True)

    # Softmax
    E = np.exp(S)
    P = E / E.sum(axis=-1, keepdims=True)
    
    # Weighted average output
    # (B,h,T,T) @ (B,h,T,d_head) -> (B,h,T,d_head) = (2,2,4,4)
    O = P @ V

    # Transpose back, reshape: -> (B,T,h,d_head) -> (B,T,d)
    O = O.transpose(0,2,1,3).reshape(B,T,d)

    # Multiply Wo to merge split heads back into a single vector,
    # and to allow the attention heads to communicate with each other.
    # That's why we need to reshape back to (B,T,d)
    O = O @ Wo

    return O, (X, Q, K, V, P, Wq, Wk, Wv, Wo, scale)

## Position-wise MLP

- An **MLP (Multi-Layer Perceptron)**, or Feed-Forward Network (FFN) is a *fully-connected* network block.
- **position-wise** means it processes the feature vector of every single token independently with exact same weights across all positions.
- Attention moves info across positions (token-to-token mixing). MLP processes info **within a position**.
- **Rank Collapse problem** (Dong et al. (2021)), if you stack attention layers without MLPs and residual connections, the tokens lose rank doubly exponentially - meaning all word vectors collapse and become virtually identical to each other. The MLP injects non-linear representations to prevent this.

Implementation:

1. First Linear projection: W1 expands hidden dimension from d -> 4d (8 -> 32)
2. GELU Activation
3. Second Linear projection: W2 contracts the dimension from 4d -> d (32 -> 8)

$$
\text{GELU}(x) \approx \frac{1}{2} x \left(
    1 + \tanh \left( \sqrt{\frac{2}{\pi}} * (x + 0.044715 x^3) \right)
\right)
$$

In [3]:
def gelu(x):
    return .5 * x * (1.0 + np.tanh(np.sqrt(2.0/np.pi) * (x + .044715*x**3)))

In [4]:
def mlp_forward(O, W1, b1, W2, b2):
    """
    Position-wise MLP Layer
    
    Shapes:
        - O:    (B,T,d) -> (2,4,8)
        - W1:   (d, 4d) -> (8,32)
        - b1:   (4d,)   -> (32,)
        - W2:   (4d, d) -> (32,8)
        - b2:   (d,)    -> (8,)
    """
    # First layer
    H1 = O @ W1 + b1

    # Activation
    A1 = gelu(H1)

    # Second layer
    H2 = A1 @ W2 + b2

    return H2, (O, H1, A1, W1, W2)

## Residual Connection and Residual Stream

Instead of treating layers as a strict pipeline where features are transformed sequentially, modern ML treats the model as a **Residual Stream** - a shared bandwidth where layers read and write info additively.

Old school: $x_{l+1} = f{x_l}$, and in backprop, gradients pass through every layer's weight matrix. If weights are small -> **Vanishing gradients**.

**Residual stream**:

$$
x_{l+1} = x_l + f(x_l)
$$

when we take the derivative (Jacobian):

$$
\frac{\partial x_{l+1}}{\partial x_l} = I + J_f
$$

and when we chain this multiplication across L layers:

$$
(I + J_L)...(I + J_1) = I + \sum_{l=1}^L J_l + \text{ some product}
$$

The stand-alone I at front prevents the gradient shrinks down to zero.

If you keep adding values to a single stream over and over, the numbers will naturally drift and grow larger. If the variance of your vectors blows up, your activations will clip, and training will collapse.

To keep the stream balanced, architectures like GPT-2 apply a structural normalization step. They scale down the initialization weights of the output projections by a factor of:

$$
\frac{1}{\sqrt{2L}}
$$

In [5]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Layer Normalisation over the last dimension (d).
    Normalises each token embedding independently.
    """
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

In [6]:
x = np.array([2, 4, 6, 8])
print(layer_norm(x, 1, 0,eps=0))

[-1.34164079 -0.4472136   0.4472136   1.34164079]


Architecture:

```
Multi-head Attention -> Position-wise MLP
```

So we build:
- first sublayer: multi-head attention - <font color=seagreen>DONE</font>
- second sublayer: position-wise mlp - <font color=seagreen>DONE</font>
- combine the two into a single block

In [22]:
def transformer_block_forward(X, params, h, d_head):
    """
    Executes one full layer over the Residual Stream 'X'.
    Following Pre-LN convention:
        X_next = X + Attn(LN(X))
        X_next = X_next + MLP(LN(X_next))
    """
    # Unpack weights and biases from the param dict.
    ln1_g, ln1_b = params['ln1_gamma'], params['ln1_beta']
    Wq, Wk, Wv, Wo = params['q_proj'], params['k_proj'], params['v_proj'], params['o_proj']

    ln2_g, ln2_b = params['ln2_gamma'], params['ln2_beta']
    W1, b1, W2, b2 = params['mlp_w1'], params['mlp_b1'], params['mlp_w2'], params['mlp_b2']

    # Attention block - Read LN -> Process -> Add
    X_norm1 = layer_norm(X, ln1_g, ln1_b)
    attn_out, attn_cache = attention_forward(X_norm1, Wq, Wk, Wv, Wo, h, d_head)
    X = X + attn_out

    # Position-wise MLP block - Read LN -> Process -> Add
    X_norm2 = layer_norm(X, ln2_g, ln2_b)
    mlp_out, mlp_cache = mlp_forward(X_norm2, W1, b1, W2, b2)
    X = X + mlp_out

    return X, (attn_cache, mlp_cache)

## Positional Embedding

By default, the core operations of a Transformer (Matrix Multiplications, Softmax, LayerNorm) do not have any concept of time or sequence. If you scramble the order of the words in a sentence, a standard attention block will output the exact same representations, just scrambled in the exact same order. This property is called **permutation equivariance**.

The simplest method is learned absolute positional embedding. You create a lookup table of weights where every coordinate index (Seat 0, Seat 1, Seat 2...) gets its own unique vector of size d.

In [8]:
# Setup for your (B, T, d) = (2, 4, 8) example
B, T, d = 2, 4, 8
block_size = 4  # Maximum context window length

# 1. Standard token embeddings from your vocabulary
tok_emb = np.random.randn(B, T, d) 

# 2. Positional lookup table (stamped seat numbers)
pos_table = np.random.randn(block_size, d) 
pos_emb = pos_table[:T, :] # Slice up to current sequence length T -> Shape: (4, 8)

# 3. Combine them via broadcasting: (2, 4, 8) + (4, 8) -> (2, 4, 8)
X = tok_emb + pos_emb 

The limitation of this method is right there in the code. If your pos_table is allocated for a block_size of 4, the model completely breaks if you pass a sentence of 5 words because it has no weight vector allocated for Seat Index 4. During inference, you must strictly truncate/crop your text to fit this maximum window size.

In [9]:
class LearnedPositionalEmbedding:

    def __init__(self, block_size, d):
        self.block_size = block_size
        self.d = d
        self.pos_table = np.random.randn(block_size, d) * .02
    
    def forward(self, tok_emb):
        B, T, d = tok_emb.shape
        pos_emb = self.pos_table[:T, :] # (T, d)
        return tok_emb + pos_emb    # (B,T,d) + (T,d) -> (B,T,d)

The modern approach is **Sinusoidal positional embeddings**. Clock hands moving at different speeds.
- The first channel cycles incredibly fast (like a second hand).
- The next channels cycle progressively slower (minute hand, hour hand...).

For a 4-token sequence with d=8, the frequencies decrease geometrically. At position p=1, your vector channels are calculated using alternating sine and cosine functions:
`[sin(1), cos(1), sin(0.1), cos(0.1), sin(0.01), cos(0.01), sin(0.001), cos(0.001)]`
Which yields the exact array snippet noted in your text: `[0.841, 0.540, 0.100, 0.995, ...]`

In [10]:
def get_sinusoidal_embeddings(T, d):
    """
    Generates static Sinusoidal Positional Embeddings.
    Frequencies mimic clock hands moving at different geometric speeds.
    """
    # Create a positional array: (T, 1) -> [[0], [1], [2], [3]]
    positions = np.arange(T)[:, np.newaxis]

    # Compute the geometric frequencies for the channels: (d//2,)
    # Use 2 * (i // 2) step to assign the same frequency to the sine/cosine pair.
    div_term = 10000.0 ** (np.arange(0,d,2) / d)

    # Init the static embedding matrix
    pe = np.zeros((T, d))

    # Fill even dimensions with sines, odd dimensions with cosines
    pe[:, 0::2] = np.sin(positions / div_term)
    pe[:, 1::2] = np.cos(positions / div_term)

    return pe

## The GPT pipeline

Add the generated matrix directly to your token embeddings before passing the combined tensor into your sequential Transformer blocks.

In [11]:
def gpt_forward(idx, token_embedding_weights, transformer_layers, final_ln_params, lm_head_weights, h, d_head):
    """
    Full GPT Forward pipeline.
    
    Arguments:
    - idx:                     Token integer IDs of shape (B, T)
    - token_embedding_weights: Vocabulary embedding matrix of shape (V, d)
    - transformer_layers:      List of dictionaries containing weights for each block
    - final_ln_params:         Dictionary with 'gamma' and 'beta' for the final LayerNorm
    - lm_head_weights:         Linear projection matrix of shape (d, V) to get word logits
    """
    B, T = idx.shape
    V, d = token_embedding_weights.shape

    # 1. Token Lookup: Map discrete token indices to dense vectors (B, T) -> (B, T, d)
    # Using advanced indexing to fetch vectors across vocabulary
    X = token_embedding_weights[idx]

    # 2. Generate static sinusoidal positional embeddings matrix -> (T, d)
    pos_embeddings = get_sinusoidal_embeddings(T, d)

    # 3. Inject Spatial info into the Residual Stream -> (B, T, d) + (T, d)
    # NumPy broadcasts the (T, d) matrix perfectly across the batch axis
    X_stream = X + pos_embeddings

    # 4. Run sequentially through stacked transformer layers (The Residual Stream)
    layer_caches = []
    for layer_params in transformer_layers:
        X_stream, block_cache = transformer_block_forward(X_stream, layer_params, h, d_head)
        layer_caches.append(block_cache)
    
    # 5. Final Layer Normalization: Standard GPT practice before reading logits
    ln_f_g, ln_f_b = final_ln_params['gamma'], final_ln_params['beta']
    X_stream = layer_norm(X_stream, ln_f_g, ln_f_b)

    # 6. LM Head Projection: Map hidden channels back into vocabulary logits
    # (B, T, d) @ (d, V) -> (B, T, V)
    logits = X_stream @ lm_head_weights
    
    # Returns final vocabulary logits and state cache
    return logits, (idx, X, pos_embeddings, layer_caches, final_ln_params, lm_head_weights)

## The PyTorch twin

In [12]:
from transformer.gpt import GPT

V = 50257
d = 128
h = 4
L = 4
dff = 512
block = 128
dropout = .1

model = GPT(V=V, d=d, h=h, dff=dff, L=L, block=block, dropout=dropout)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

Total params: 13,657,088


Twin check between NumPy and PyTorch with a tolerance below 1e-13. Compute the absolute maximum difference (often called the infinity norm or `max_error`) between the two outputs.

In [13]:
B, T, V, d = 2, 4, 10, 16  # Small dimensions for pure mathematical alignment
h = 2                      # Number of heads
d_head = d // h            # Head size
L = 2                      # Number of layers
dff = 32                   # Hidden size for MLP

In [14]:
# Initialize PyTorch module in double precision
torch_model = GPT(V=V, d=d, h=h, dff=dff, L=L, block=T, dropout=0.0).to(torch.float64)
torch_model.eval() # XXX: Deactivate any dropout layer random masking!
torch.set_default_dtype(torch.float32)

# Mock Input IDs (B, T)
np_idx = np.random.randint(0, V, size=(B, T))
torch_idx = torch.tensor(np_idx, dtype=torch.long)

In [15]:
# Extract token embedding weights
token_embedding_weights = torch_model.tok.weight.detach().numpy()

# Extract LM Head weights (PyTorch stores as (V, d), NumPy code expects (d, V))
lm_head_weights = torch_model.head.weight.detach().numpy().T

# Extract Final LayerNorm parameters
final_ln_params = {
    'gamma': torch_model.ln_f.weight.detach().numpy(),
    'beta': torch_model.ln_f.bias.detach().numpy()
}

In [19]:
# Map Stacked Transformer Block Weights
# NOTE: Modify sub-keys if your transformer.block weights use different naming conventions
transformer_layers = []
for block in torch_model.blocks:
    layer_dict = {
        # PyTorch Attention weights are stored transposed (out_features, in_features)
        'q_proj': block.attn.wq.weight.detach().numpy().T,
        'k_proj': block.attn.wk.weight.detach().numpy().T,
        'v_proj': block.attn.wv.weight.detach().numpy().T,
        'o_proj': block.attn.wo.weight.detach().numpy().T,
        
        # MLP Weights
        'mlp_w1': block.mlp.w1.weight.detach().numpy().T,
        'mlp_w2': block.mlp.w2.weight.detach().numpy().T,
        
        # LayerNorm Weights & Biases
        'ln1_gamma': block.ln1.weight.detach().numpy(),
        'ln1_beta': block.ln1.bias.detach().numpy(),
        'ln2_gamma': block.ln2.weight.detach().numpy(),
        'ln2_beta': block.ln2.bias.detach().numpy(),
    }
    # Add biases if they exist in your Block definition
    if block.attn.wq.bias is not None:
        layer_dict['q_bias'] = block.attn.wq.bias.detach().numpy()
        layer_dict['k_bias'] = block.attn.wk.bias.detach().numpy()
        layer_dict['v_bias'] = block.attn.wv.bias.detach().numpy()
        layer_dict['o_bias'] = block.attn.wo.bias.detach().numpy()
    if block.mlp.w1.bias is not None:
        layer_dict['mlp_b1'] = block.mlp.w1.bias.detach().numpy()
        layer_dict['mlp_b2'] = block.mlp.w2.bias.detach().numpy()
        
    transformer_layers.append(layer_dict)

In [23]:
# Run PyTorch Pipeline
with torch.no_grad():
    torch_logits, _ = torch_model(torch_idx)
    torch_logits_np = torch_logits.numpy()

# Run NumPy Pipeline
# (Ensure your supporting functions like get_sinusoidal_embeddings use float64)
numpy_logits, _ = gpt_forward(
    idx=np_idx,
    token_embedding_weights=token_embedding_weights,
    transformer_layers=transformer_layers,
    final_ln_params=final_ln_params,
    lm_head_weights=lm_head_weights,
    h=h,
    d_head=d_head
)

In [24]:
# Compute Absolute Max Discrepancy
max_diff = np.max(np.abs(numpy_logits - torch_logits_np))
print(f"Maximum discrepancy between implementations: {max_diff:.2e}")

assert max_diff < 1e-13, f"Twin Check FAILED. Discrepancy is {max_diff:.2e}"
print("Twin Check PASSED! Both architectures match within absolute float64 tolerance limit.")

Maximum discrepancy between implementations: 4.15e-07


AssertionError: Twin Check FAILED. Discrepancy is 4.15e-07

## Training: what the loss number means

Imagine you are playing a game where you must guess the next character in a book.

- **Total Ignorance** (Random Guessing): If you know nothing about English and the vocabulary has 65 characters (letters, punctuation, spaces), you are drawing blindly from a hat of 65 slips.
- **The Bigram Baseline**: If you look at only the immediate previous character (e.g., seeing a 'q' and knowing 'u' usually follows), you throw away the bad slips. Your pool of candidates shrinks from 65 down to an effective ~12.
- **The Transformer** (Contextual Attention): As attention layers learn grammar, context, and plot, the pool shrinks further. Perplexity is simply the size of that remaining pool of equivalent guesses.

For a uniform distribution over 65 classes, $p_i = \frac{1}{65}$. The loss is:

$$
-\ln(\frac{1}{65}) = \ln(65) \approx 4.17
$$

Perplexity is $e^{\ln(65)} = 65$.

A loss of $2.47$ yields $e^2.47 \approx 11.8$. Until your model drops below this threshold, your multi-head attention blocks are adding computational overhead without outperforming a simple lookup table of past characters.

With batch size $B = 64$, and block size $T = 128$, each step processes $64 * 128 = 8192$ tokens. At 5,000 steps, you process 40.96 million tokens. Since Tiny Shakespeare is roughly 1.1 million characters, 5,000 steps equals roughly 37 full passes over the dataset.

At 37 passes, a multi-million parameter model has enough capacity to simply memorise the training sequences word-for-word rather than learning generalisable language patterns. This is why validation loss starts creeping back up and why dropout and weight decay are non-negotiable.

In [ ]:
# %run ../scripts/bigram_run.py

```
807936
    0  train 4.1709  val 4.1710  29s
  250  train 2.4999  val 2.4934  321s
  500  train 2.2013  val 2.2118  611s
  750  train 1.9471  val 2.0136  905s
 1000  train 1.7825  val 1.9012  1195s
 1250  train 1.6692  val 1.8223  1484s
 1500  train 1.5976  val 1.7728  1778s
 1750  train 1.5499  val 1.7352  2067s
 2000  train 1.5052  val 1.6889  2360s
 2250  train 1.4695  val 1.6508  2653s
 2500  train 1.4516  val 1.6385  2942s
 2750  train 1.4286  val 1.6277  3230s
 3000  train 1.4090  val 1.6091  3517s
 3250  train 1.3926  val 1.5955  3804s
 3500  train 1.3882  val 1.5820  4098s
 3750  train 1.3743  val 1.5807  4390s
 4000  train 1.3620  val 1.5588  4683s
 4250  train 1.3556  val 1.5629  4978s
 4500  train 1.3497  val 1.5601  5273s
 4750  train 1.3482  val 1.5532  5570s
 5000  train 1.3418  val 1.5584  5865s

And thine noble of this father's cun far't,
And Warwick's remembers with but-stacte:
Hengean my spiring slare now her majesty.

KING RICHARD II:
Wellow most on own it, but callent dismistrue galow.

HEMIONE:
It till me. Were, where tongue young heard, come,
Being our consencene!
Farewell, why, thou is to make like on comas,
Grandforth thought thou from in with worlds the stear
Against with a sew his constron devanced?
And I cone a lifes; thereof the nose,
And I spechanted such priving them: He g
```